In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

# Assuming 'path' variable exists from the kagglehub block
# We look for the CSV file inside the downloaded directory
csv_file_path = os.path.join(path, "Q3_data.csv")

# If the file is directly in the path, this works.
# If the path points to a folder containing the CSV, we might need to search for it:
if not os.path.exists(csv_file_path):
    for root, dirs, files in os.walk(path):
        for file in files:
            if file == "Q3_data.csv":
                csv_file_path = os.path.join(root, file)

df = pd.read_csv(csv_file_path)
print("Data loaded successfully.")

In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Check for missing values
print("Missing values before:\n", df.isnull().sum())

# Fill missing numerical values with the median (a common strategy)
# If there are specific categorical columns, we would fill them with mode or a placeholder
numerical_cols = df.select_dtypes(include=['number']).columns
df[numerical_cols] = df[numerical_cols].fillna(df[numerical_cols].median())

# Verify
print("Missing values after:\n", df.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:
duplicates = df.duplicated().sum()
print(f"Duplicates found: {duplicates}")

if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicates to remove.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

# Identify categorical columns (object type)
categorical_cols = df.select_dtypes(include=['object']).columns

if len(categorical_cols) > 0:
    le = LabelEncoder()
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col])
    print(f"Encoded columns: {list(categorical_cols)}")
else:
    print("No categorical columns found needing encoding.")

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# We scale all features except the target (assuming target column is named 'target')
# You need to verify the exact name of your target column from Part 1 info().
# Commonly it is 'target', 'default', or 'label'. Let's assume 'target'.
target_col = 'Target'

feature_cols = [col for col in df.columns if col != target_col]

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Feature scaling applied.")
df.head()

In [ ]:
# Task 5: Write your code here:
target_counts = df[target_col].value_counts(normalize=True)
print("Target Distribution:\n", target_counts)

if target_counts.min() < 0.2:  # Threshold can vary, but < 20% is typically considered imbalanced
    print("\nConclusion: The dataset is IMBALANCED.")
else:
    print("\nConclusion: The dataset is NOT severely imbalanced.")

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
!pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier
import numpy as np

# Initialize StratifiedKFold (Correct choice for imbalanced/classification data)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
accuracy_scores = []

# Loop
fold = 1
for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Initialize CatBoost (verbose=0 suppresses training output)
    model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_state=42)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)


    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)

    f1_scores.append(f1)
    accuracy_scores.append(acc)

    print(f"Fold {fold} | F1 Score: {f1:.4f} | Accuracy: {acc:.4f}")
    fold += 1

print("-" * 30)
print(f"Average F1 Score: {np.mean(f1_scores):.4f}")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")